# Notebook 11: delta learning

Instead of predicting a molecule's activity directly, train on **pairs** of similar
molecules and predict the difference between them. At prediction time, find each
compound's nearest neighbours among the molecules we have measured, predict the
difference, and add it to the neighbour's known value.

The reason to expect this to work here is a measured property of this challenge rather
than a general argument. The organisers built the test set by purchasing Enamine
analogues of the top training hits, and we measured the consequence: blind compounds sit
at median Tanimoto 0.578 from our training set, with 97% above 0.4 and 40% above 0.6.
Almost every blind compound has a close relative whose activity we already know. Direct
prediction throws that away; delta prediction anchors on it.

It is also a completely different error structure from everything else we have, which is
the property that has actually paid in this blend twice.

**No GPU needed.** This is LightGBM on CPU, so it can run alongside a GPU notebook.

Fold integrity matters and is handled explicitly: a training pair for fold k uses two
molecules both drawn from that fold's training rows, never one from the validation side.

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/tiny_6387286b00.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install rdkit lightgbm

In [ ]:
import json, time, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import lightgbm as lgb
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy')
G=np.load(CACHE/'scaffold_groups.npy', allow_pickle=True)
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

names=[d[0] for d in Descriptors._descList]
calc=MoleculeDescriptors.MolecularDescriptorCalculator(names)
gen=AllChem.GetMorganGenerator(radius=2, fpSize=2048)

def featurise(sm):
    X=np.zeros((len(sm), 2048+len(names)), np.float32); fps=[]
    for i,s in enumerate(sm):
        m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
        if m is None:
            fps.append(None); continue
        fps.append(gen.GetFingerprint(m))
        X[i,:2048]=gen.GetFingerprintAsNumPy(m)
        X[i,2048:]=np.nan_to_num(np.asarray(calc.CalcDescriptors(m), np.float32),
                                 nan=0.0, posinf=0.0, neginf=0.0)
    return X, fps

t0=time.time()
XTR, FTR = featurise(SM_TR)
XTE, FTE = featurise(SM_TE)
print(f'featurised {XTR.shape} train, {XTE.shape} test ({time.time()-t0:.0f}s)')
RESULTS=[]

## Pair construction and the delta model

A pair is two molecules above a similarity floor. The features are what changed between
them plus where they started, and the target is the change in activity.

In [ ]:
SIM_FLOOR = 0.40      # blind compounds sit above this against training 97% of the time
MAX_PAIRS = 150000    # sampled if a fold produces more
K_NEIGHBOURS = 5      # how many anchors each prediction averages over
PARAMS=dict(objective='regression', learning_rate=0.05, num_leaves=63,
            min_data_in_leaf=40, feature_fraction=0.15, bagging_fraction=0.8,
            bagging_freq=1, lambda_l2=1.0, verbosity=-1, num_threads=4)
ROUNDS=500
rng=np.random.default_rng(0)

def sim_matrix(a_fps, b_fps):
    return np.array([DataStructs.BulkTanimotoSimilarity(f, b_fps) for f in a_fps],
                    dtype=np.float32)

def pair_features(i_idx, j_idx, Xi, Xj, sims):
    """What changed, where it started, and how close the two are."""
    return np.hstack([Xi[i_idx] - Xj[j_idx], Xj[j_idx][:, 2048:], sims[:, None]]).astype(np.float32)

MIN_PAIRS = 20000
def build_pairs(anchor_rows, target_rows, y, cap):
    """Pairs (t, a) above a similarity floor; target is y[t] - y[a].

    The floor drops until there are enough pairs to fit on. At 0.40 a thousand-odd
    anchors yield only a few thousand pairs, which is thin for a model this wide, so
    taking more distant pairs is better than starving it."""
    S = sim_matrix([FTR[i] for i in target_rows], [FTR[i] for i in anchor_rows])
    for floor in [SIM_FLOOR, 0.35, 0.30, 0.25, 0.20, 0.15]:
        ti, ai = np.where(S >= floor)
        keep = target_rows[ti] != anchor_rows[ai]
        ti, ai = ti[keep], ai[keep]
        if len(ti) >= MIN_PAIRS or floor == 0.15:
            break
    if len(ti) > cap:
        sel = rng.choice(len(ti), cap, replace=False); ti, ai = ti[sel], ai[sel]
    return target_rows[ti], anchor_rows[ai], S[ti, ai], floor

def predict_delta(model, query_rows, query_X, query_fps, anchor_rows, y):
    """Average over each query's K nearest anchors, after adding the predicted delta."""
    S = sim_matrix(query_fps, [FTR[i] for i in anchor_rows])
    out = np.full(len(query_rows), np.nan)
    order = np.argsort(-S, axis=1)[:, :K_NEIGHBOURS]
    for n in range(len(query_rows)):
        a = anchor_rows[order[n]]
        f = np.hstack([query_X[n] - XTR[a], XTR[a][:, 2048:],
                       S[n, order[n]][:, None]]).astype(np.float32)
        out[n] = np.mean(model.predict(f) + y[a])
    return out

## Smoke test, about a minute

In [ ]:
iso='CYP3A4'
y=meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
dev=np.flatnonzero(np.isfinite(y) & ~hold)[:400]
t,a,s,fl = build_pairs(dev[:300], dev[:300], y, 20000)
print(f'pairs built: {len(t)} at floor {fl:.2f} | similarity '
      f'{s.min():.2f} to {s.max():.2f}')
assert len(t)>100, 'too few pairs; lower SIM_FLOOR'
F = pair_features(t, a, XTR, XTR, s)
print('pair features:', F.shape, '| finite', bool(np.isfinite(F).all()))
m = lgb.train(PARAMS, lgb.Dataset(F, y[t]-y[a]), 60)
p = predict_delta(m, dev[300:400], XTR[dev[300:400]], [FTR[i] for i in dev[300:400]],
                  dev[:300], y)
print('predictions:', p.shape, '| finite', bool(np.isfinite(p).all()),
      '| rho %.4f'%spearmanr(y[dev[300:400]], p).statistic)
print('\nSMOKE TEST PASSED')

## Full run

Every fold trains pairs drawn only from its own training rows, so no validation molecule
appears on either side of a training pair.

In [ ]:
t0=time.time()
for iso in ISO:
    y=meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    obs=np.isfinite(y)
    dev=np.flatnonzero(obs & ~hold); ho=np.flatnonzero(obs & hold)
    oof=np.full(len(dev), np.nan)
    for f,(tr,va) in enumerate(GroupKFold(5).split(dev, dev, G[dev])):
        A=dev[tr]
        t,a,s,fl = build_pairs(A, A, y, MAX_PAIRS)
        m = lgb.train(PARAMS, lgb.Dataset(pair_features(t,a,XTR,XTR,s), y[t]-y[a]), ROUNDS)
        q=dev[va]
        oof[va]=predict_delta(m, q, XTR[q], [FTR[i] for i in q], A, y)
    t,a,s,fl = build_pairs(dev, dev, y, MAX_PAIRS)
    m = lgb.train(PARAMS, lgb.Dataset(pair_features(t,a,XTR,XTR,s), y[t]-y[a]), ROUNDS)
    hp = predict_delta(m, ho, XTR[ho], [FTR[i] for i in ho], dev, y)
    allr=np.concatenate([dev,ho])
    t,a,s,fl = build_pairs(allr, allr, y, MAX_PAIRS)
    m2 = lgb.train(PARAMS, lgb.Dataset(pair_features(t,a,XTR,XTR,s), y[t]-y[a]), ROUNDS)
    tp = predict_delta(m2, np.arange(len(SM_TE)), XTE, FTE, allr, y)
    np.savez_compressed(OUT/f'delta_{iso}.npz', oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
    r=float(spearmanr(y[dev],oof).statistic); rh=float(spearmanr(y[ho],hp).statistic)
    RESULTS.append(dict(iso=iso, dev_spearman=r, holdout_spearman=rh))
    json.dump(RESULTS, open(OUT/'results_nb11.json','w'), indent=2)
    print(f'{iso}: dev rho {r:.4f} | holdout rho {rh:.4f} | {len(t)} pairs at floor {fl:.2f}  ({time.time()-t0:.0f}s)', flush=True)
print('\nfor comparison, the arms already in the blend, dev:')
print('  TabPFN raw  0.5881 0.7060 0.4448 0.8115')
print('  multitask   0.5577 0.6935 0.4515 0.8022')
print('It does not need to beat them. It needs to be close and to disagree.')

import shutil
from google.colab import files
shutil.make_archive('/content/nb11_results','zip',OUT)
files.download('/content/nb11_results.zip')